# Stage 2 — fusion comparison
Compare DRS-only, residual (main), and optional FiLM experiments on the same LOSO folds. This supersedes `analyze_fusion_strategy` and supports any number of experiment directories.

In [1]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').is_file():  # VS Code may start in notebooks/
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Open this notebook from final_refactored or its notebooks folder.')
os.chdir(ROOT)
SRC_DIR = ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))
SUBPROCESS_ENV = os.environ.copy()
SUBPROCESS_ENV['PYTHONPATH'] = str(SRC_DIR) + os.pathsep + SUBPROCESS_ENV.get('PYTHONPATH', '')
ARTIFACTS = Path('artifacts/stage2')
OUTPUT = Path('results/tables/stage2/fusion_comparison')
EXPERIMENTS = {
    ARTIFACTS / 'hc_baseline_scratch_main': 'DRS-only',
    ARTIFACTS / 'hc_fusion_dtof_actual_pretrained_finetune_residual_gated_main': 'Residual',
    ARTIFACTS / 'hc_fusion_dtof_actual_pretrained_finetune_film_comparison': 'FiLM',
}
AVAILABLE_EXPERIMENTS = {path: label for path, label in EXPERIMENTS.items() if (path / 'loso_results.csv').is_file()}
AVAILABLE_EXPERIMENTS

{PosixPath('artifacts/stage2/hc_baseline_scratch_main'): 'DRS-only',
 PosixPath('artifacts/stage2/hc_fusion_dtof_actual_pretrained_finetune_residual_gated_main'): 'Residual',
 PosixPath('artifacts/stage2/hc_fusion_dtof_actual_pretrained_finetune_film_comparison'): 'FiLM'}

## Build the overall comparison
The primary paired quantity is ΔRMSE = fusion RMSE − DRS-only RMSE for the same held-out subject; negative values favor fusion.

In [2]:
baseline_path = next(path for path, label in EXPERIMENTS.items() if label == 'DRS-only')
if baseline_path not in AVAILABLE_EXPERIMENTS:
    raise FileNotFoundError(f'Run the DRS-only baseline LOSO experiment first. Missing: {baseline_path / "loso_results.csv"}')
if len(AVAILABLE_EXPERIMENTS) < 2:
    raise FileNotFoundError('At least one completed fusion experiment is required in addition to DRS-only.')
command = [sys.executable, '-m', 'subject_nirs.stage2.comparison', '--output-dir', str(OUTPUT), '--no-pdf']
for path, label in AVAILABLE_EXPERIMENTS.items():
    command.extend(['--experiment', f'{path}={label}'])
subprocess.run(command, cwd=ROOT, env=SUBPROCESS_ENV, check=True)

Loaded DRS-only               target=hc   folds=154 from /media/md703/PNY_Gen4_4TB/YC/Research/final_refactored/artifacts/stage2/hc_baseline_scratch_main
Loaded FiLM                   target=hc   folds=118 from /media/md703/PNY_Gen4_4TB/YC/Research/final_refactored/artifacts/stage2/hc_fusion_dtof_actual_pretrained_finetune_film_comparison
Loaded Residual               target=hc   folds=154 from /media/md703/PNY_Gen4_4TB/YC/Research/final_refactored/artifacts/stage2/hc_fusion_dtof_actual_pretrained_finetune_residual_gated_main



Experiment summary (median RMSE / median |Bias|):
    HC | DRS-only                 | n=154 | RMSE=14.034 | |Bias|=8.9657
    HC | FiLM                     | n=118 | RMSE=13.14 | |Bias|=8.3243
    HC | Residual                 | n=154 | RMSE=12.493 | |Bias|=7.4742

Paired comparison vs DRS only:
    HC | FiLM                     | median ΔRMSE=-0.56408 | Improved=58.5% | Wilcoxon p=0.0232
    HC | Residual                 | median ΔRMSE=-0.7652 | Improved=63.0% | Wilcoxon p=7.66e-05

Conditional effect by baseline difficulty:
    HC | FiLM                     | Q4-Q1 mean ΔRMSE=-4.5881 [-6.4775, -2.8009]
    HC | Residual                 | Q4-Q1 mean ΔRMSE=-4.2999 [-5.7882, -2.899]

Saved analysis to: /media/md703/PNY_Gen4_4TB/YC/Research/final_refactored/results/tables/stage2/fusion_comparison


CompletedProcess(args=['/home/md703/.conda/envs/yc_ae/bin/python', '-m', 'subject_nirs.stage2.comparison', '--output-dir', 'results/tables/stage2/fusion_comparison', '--no-pdf', '--experiment', 'artifacts/stage2/hc_baseline_scratch_main=DRS-only', '--experiment', 'artifacts/stage2/hc_fusion_dtof_actual_pretrained_finetune_residual_gated_main=Residual', '--experiment', 'artifacts/stage2/hc_fusion_dtof_actual_pretrained_finetune_film_comparison=FiLM'], returncode=0)

In [3]:
import pandas as pd
pd.read_csv(OUTPUT / 'experiment_summary.csv')

,target,experiment,n_subjects,RMSE_mean,RMSE_std,RMSE_median,RMSE_IQR,MAE_mean,MAE_std,MAE_median,...,Bias_median,Bias_IQR,AbsBias_mean,AbsBias_std,AbsBias_median,AbsBias_IQR,ErrorStd_mean,ErrorStd_std,ErrorStd_median,ErrorStd_IQR
0,hc,DRS-only,154,14.893295,5.596816,14.033874,7.368020,12.436579,4.893511,11.626775,...,1.071402,18.130057,9.638630,6.573854,8.965663,10.261182,10.544010,2.440405,10.364954,3.744043
1,hc,FiLM,118,13.992386,5.222937,13.139883,7.693276,11.672620,4.544516,10.786510,...,0.429174,16.185642,8.796501,6.287635,8.324304,9.798976,10.016098,2.446533,10.000870,3.811767
2,hc,Residual,154,13.911258,5.414286,12.492620,7.310490,11.614910,4.754929,10.447774,...,-0.362965,14.934109,8.640627,6.401782,7.474206,9.960554,10.045180,2.531027,9.952631,3.753961
